# Official workload tail scaling
The canonical input is `official_token_summary.csv`; `motivation_tail_scaling.csv` is retained only as a historical compatibility name.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)
frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists(): frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    order = {'short': 1, 'medium': 2, 'long': 3}
    df['scale_id'] = df['scale'].map(order)
    view = df.groupby(['suite', 'mode', 'scale', 'scale_id'], as_index=False)[['total_tokens_p95', 'total_tokens_p99']].mean().sort_values('scale_id')
    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8), dpi=300, sharey=True)
    for ax, metric, title in zip(axes, ['total_tokens_p95', 'total_tokens_p99'], ['p95', 'p99']):
        for (suite, mode), group in view.groupby(['suite', 'mode']):
            ax.plot(group['scale_id'], group[metric], marker='o', linewidth=1.0, label=f'{suite}:{mode}')
        ax.set_title(title + ' tokens', fontsize=8)
        ax.set_xticks([1, 2, 3], ['short', 'medium', 'long'])
        ax.set_xlabel('Trajectory length (# calls)', fontsize=8)
        ax.tick_params(labelsize=7)
    axes[0].set_ylabel('Tokens', fontsize=8)
    axes[1].legend(fontsize=5.5, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Motivation-Tail-Scaling.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Motivation-Tail-Scaling.pdf', bbox_inches='tight')
else:
    print('No official summary found; run the official benchmark first.')
# Compatibility filename: motivation_tail_scaling.csv; official_token_summary.csv is authoritative.
